# Leçon 05 - RAG Agentique


## Configuration

Ce notebook démontre le modèle Agentic RAG (Retrieval-Augmented Generation) en utilisant le Microsoft Agent Framework.

**Prérequis :**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — le point de terminaison de votre service Azure AI Search
- `AZURE_SEARCH_API_KEY` — la clé API de votre service Azure AI Search
- Déploiement Azure OpenAI configuré via des variables d'environnement
- Azure CLI authentifié (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Qu'est-ce que Agentic RAG ?

Le RAG traditionnel suit une pipeline fixe : récupérer des documents, puis générer une réponse. **Agentic RAG** va plus loin en donnant à l'agent l'autonomie de décider **quand** et **comment** récupérer l'information.

Avec Agentic RAG, l'agent peut :
- **Décider** si une récupération est nécessaire avant de répondre à une question
- **Choisir** quelle source de données ou outil interroger
- **Évaluer** les résultats récupérés et effectuer des récupérations supplémentaires si la première tentative est insuffisante
- **Combiner** les informations de plusieurs étapes de récupération en une réponse cohérente

Cela rend l'agent plus flexible et précis comparé à une pipeline statique de récupération-puis-génération.


## Création d’un outil de recherche

Dans Agentic RAG, les sources de données externes sont emballées en tant que **outils** que l’agent peut invoquer à la demande. Cela permet à l’agent de considérer la récupération comme une autre action qu’il peut effectuer, plutôt qu’une étape obligatoire.

Nous définissons ci-dessous une base de connaissances sur les voyages et la rendons accessible en tant qu’outil que l’agent peut appeler pour rechercher des informations sur une destination.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Création de l'agent RAG

Nous créons maintenant un agent qui est instruit pour **toujours récupérer des informations avant de répondre**. L'agent utilise l'outil `search_travel_knowledge` pour ancrer ses réponses dans la base de connaissances plutôt que de se fier à ses propres données d'entraînement.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Recherche itérative — Le modèle Maker-Checker

Un avantage clé d’Agentic RAG est la **recherche itérative**. L’agent peut effectuer plusieurs tours de recherche pour vérifier, affiner ou étendre ses découvertes initiales — similaire à un workflow "maker-checker" :

1. **Étape Maker** : L’agent récupère les informations initiales et rédige une réponse.
2. **Étape Checker** : L’agent effectue des recherches supplémentaires pour vérifier les détails ou combler les lacunes.

Ci-dessous, on pose à l’agent une question qui nécessite de comparer plusieurs destinations, ce qui l’incite à effectuer plusieurs recherches.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Résumé

Dans cette leçon, vous avez appris comment construire un système **Agentic RAG** en utilisant le Microsoft Agent Framework :

- **Agentic RAG** permet aux agents de décider de manière autonome quand récupérer des informations, rendant la récupération dynamique plutôt que fixe.
- **Outils comme sources de données** : les bases de connaissances externes (comme Azure AI Search) sont encapsulées en tant qu'outils que l'agent peut invoquer.
- **Récupération itérative** : le modèle maker-checker permet à l'agent d'effectuer plusieurs tours de récupération — rechercher, vérifier et affiner — avant de produire une réponse finale.

En production, vous remplaceriez la base de connaissances en mémoire `TRAVEL_KNOWLEDGE_BASE` par un véritable index Azure AI Search pour gérer la récupération de documents de voyage à grande échelle.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Clause de non-responsabilité** :  
Ce document a été traduit à l’aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforcions d’assurer l’exactitude, veuillez noter que les traductions automatiques peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue d’origine doit être considéré comme la source officielle. Pour les informations critiques, il est recommandé de faire appel à une traduction professionnelle réalisée par un humain. Nous déclinons toute responsabilité en cas de malentendus ou de mauvaises interprétations résultant de l’utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
